# 9. Streaming Count-Min Sketch Top-K：怎样用固定内存估计查询热度并解释碰撞？

## 面试回答主线

Count-Min Sketch 用 depth 个独立哈希把每个事件映射到 width 个计数桶，查询时取对应桶计数的最小值。由于碰撞只会增加计数，append-only 场景下估计值不低于真实频次；width 控制误差，depth 控制高概率保证。CMS 本身不能枚举 key，因此 Top-K 仍需要候选集合、堆或 Space-Saving 结构。面试时我会用真实搜索词流手写稳定哈希、更新矩阵、桶索引和估计误差，并与 exact dict 基线比较。删除或负更新会破坏“只高估”保证，重复 delete 甚至污染无关 key；常见修正是时间窗轮换后从事件日志重建。生产还需考虑并发原子更新、分片 merge、counter overflow 与重热点。

## 1. 真实案例：八类搜索词组成的六十六条流事件

退款、订单和发票是高频查询，RAG 与向量相对低频。事件用确定性频次展开，便于同时拥有 exact ground truth 与流式 sketch 输入。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示查询流和估计结果
import hashlib  # 导入稳定哈希函数实现可复现的多行桶映射
frequencies = {"退款": 20, "订单": 15, "发票": 10, "密码": 7, "物流": 5, "天气": 4, "RAG": 3, "向量": 2}  # 定义八类真实搜索词的事件频次
stream = [term for term, count in frequencies.items() for _ in range(count)]  # 将聚合频次展开为逐条流事件
preview = [{"搜索词": term, "真实事件数": count} for term, count in frequencies.items()]  # 汇总输入流的业务分布
print("搜索流事件预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示八类搜索词和总流量差异
print(f"事件总数={len(stream)}")  # 输出流式处理的真实事件规模

搜索流事件预览：
[{'搜索词': '退款', '真实事件数': 20},
 {'搜索词': '订单', '真实事件数': 15},
 {'搜索词': '发票', '真实事件数': 10},
 {'搜索词': '密码', '真实事件数': 7},
 {'搜索词': '物流', '真实事件数': 5},
 {'搜索词': '天气', '真实事件数': 4},
 {'搜索词': 'RAG', '真实事件数': 3},
 {'搜索词': '向量', '真实事件数': 2}]
事件总数=66


## 2. Baseline（基线）：exact dict 保存每一个 distinct key

精确计数逐事件更新字典，结果无误差，但内存随 distinct query 数线性增长。它作为 CMS 的 ground truth 和 Top-K 对照。

In [2]:
exact_counts = {}  # 初始化随 key 数增长的精确计数字典
for term in stream:  # 按事件到达顺序处理完整搜索流
    exact_counts[term] = exact_counts.get(term, 0) + 1  # 为当前搜索词执行精确加一
exact_ranking = sorted(exact_counts.items(), key=lambda pair: (-pair[1], pair[0]))  # 按真实频次和词文本稳定排序
print("Exact dict 查询热度：")  # 标注当前输出属于精确基线
pprint(exact_ranking)  # 展示八类搜索词的真实计数与 Top-K
print({"精确counter数量": len(exact_counts), "事件总和": sum(exact_counts.values())})  # 展示精确方案的 key 级内存规模

Exact dict 查询热度：
[('退款', 20),
 ('订单', 15),
 ('发票', 10),
 ('密码', 7),
 ('物流', 5),
 ('天气', 4),
 ('RAG', 3),
 ('向量', 2)]
{'精确counter数量': 8, '事件总和': 66}


## 3. 手写 Count-Min Sketch：三行、六桶与稳定哈希

每个 query 在三行各落入一个桶。width=6 有意保留少量碰撞，使估计误差可观察；表格大小固定为 18 个 counter，与未来 distinct key 数无关。

In [3]:
class CountMinSketch:  # 定义固定宽度和深度的流式频次估计器
    def __init__(self, width, depth):  # 初始化多行计数矩阵和哈希种子数量
        self.width = width  # 保存每行桶数量控制碰撞概率
        self.depth = depth  # 保存独立哈希行数控制误差置信度
        self.table = [[0 for _ in range(width)] for _ in range(depth)]  # 分配固定数量的整数 counters
    def bucket(self, item, seed):  # 用行号作为种子计算稳定桶索引
        payload = f"{seed}|{item}".encode("utf-8")  # 组合哈希种子和搜索词字节
        digest = hashlib.blake2b(payload, digest_size=8).digest()  # 生成跨进程稳定的八字节摘要
        return int.from_bytes(digest, "big") % self.width  # 将哈希值映射到当前行的桶范围
    def add(self, item, count=1):  # 把一个流事件更新到所有哈希行
        indices = []  # 记录当前 key 在每行的桶位置
        for seed in range(self.depth):  # 遍历三个独立哈希函数
            index = self.bucket(item, seed)  # 计算当前行的目标桶
            self.table[seed][index] += count  # 将事件计数累加到对应桶
            indices.append(index)  # 保存桶索引供碰撞审计
        return indices  # 返回本次更新涉及的三行位置
    def estimate(self, item):  # 查询一个 key 的 Count-Min 频次估计
        counters = [self.table[seed][self.bucket(item, seed)] for seed in range(self.depth)]  # 读取三行对应桶的当前计数
        return min(counters), counters  # 用最小桶减轻碰撞造成的高估
cms = CountMinSketch(width=6, depth=3)  # 创建固定十八个 counter 的教学 sketch
observed_terms = set()  # 保存已出现 key 作为 Top-K 候选源
for term in stream:  # 逐事件流式更新 sketch
    cms.add(term)  # 把当前搜索事件写入三个哈希桶
    observed_terms.add(term)  # 记录该 key 可参与后续候选排序
bucket_rows = [{"搜索词": term, "三行桶": [cms.bucket(term, seed) for seed in range(cms.depth)]} for term in frequencies]  # 汇总每个真实 key 的哈希路径
print("搜索词到 CMS 哈希桶的映射：")  # 输出核心机制中间量标题
pprint(bucket_rows, sort_dicts=False)  # 展示哪些查询在某些行发生碰撞
print("最终三乘六 counter 矩阵：")  # 输出固定内存状态标题
pprint(cms.table)  # 展示全部桶计数而不是只给估计值

搜索词到 CMS 哈希桶的映射：
[{'搜索词': '退款', '三行桶': [0, 1, 0]},
 {'搜索词': '订单', '三行桶': [2, 0, 3]},
 {'搜索词': '发票', '三行桶': [5, 0, 2]},
 {'搜索词': '密码', '三行桶': [5, 5, 3]},
 {'搜索词': '物流', '三行桶': [0, 3, 4]},
 {'搜索词': '天气', '三行桶': [4, 1, 3]},
 {'搜索词': 'RAG', '三行桶': [3, 1, 4]},
 {'搜索词': '向量', '三行桶': [3, 3, 2]}]
最终三乘六 counter 矩阵：
[[25, 0, 15, 5, 4, 17], [25, 27, 0, 7, 0, 7], [20, 0, 12, 26, 8, 0]]


## 4. 逐 key 估计、每行 counter 与碰撞误差

CMS estimate 取三行 counter 最小值，因此某一行碰撞通常会被其他行抵消；如果三行都受碰撞影响则出现高估。下面保留每行 counter 以解释误差来源。

In [4]:
estimate_rows = []  # 收集八个搜索词的 sketch 估计细节
for term, exact in exact_ranking:  # 按真实热度遍历所有候选 key
    estimate, counters = cms.estimate(term)  # 读取三行桶计数和最小值估计
    estimate_rows.append({"搜索词": term, "真实频次": exact, "三行counter": counters, "CMS估计": estimate, "高估误差": estimate - exact})  # 保存逐 key 碰撞误差
print("CMS 逐搜索词估计：")  # 输出核心方案结果标题
pprint(estimate_rows, sort_dicts=False)  # 展示最小桶、真实频次和高估量

CMS 逐搜索词估计：
[{'搜索词': '退款', '真实频次': 20, '三行counter': [25, 27, 20], 'CMS估计': 20, '高估误差': 0},
 {'搜索词': '订单', '真实频次': 15, '三行counter': [15, 25, 26], 'CMS估计': 15, '高估误差': 0},
 {'搜索词': '发票', '真实频次': 10, '三行counter': [17, 25, 12], 'CMS估计': 12, '高估误差': 2},
 {'搜索词': '密码', '真实频次': 7, '三行counter': [17, 7, 26], 'CMS估计': 7, '高估误差': 0},
 {'搜索词': '物流', '真实频次': 5, '三行counter': [25, 7, 8], 'CMS估计': 7, '高估误差': 2},
 {'搜索词': '天气', '真实频次': 4, '三行counter': [4, 27, 26], 'CMS估计': 4, '高估误差': 0},
 {'搜索词': 'RAG', '真实频次': 3, '三行counter': [5, 27, 8], 'CMS估计': 5, '高估误差': 2},
 {'搜索词': '向量', '真实频次': 2, '三行counter': [5, 7, 12], 'CMS估计': 5, '高估误差': 3}]


## 5. 结果解读：CMS Top-K 需要外部候选源

这里用 observed_terms 作为有限候选集合，再按 estimate 排序。真实无限流不能保存所有 key，通常配合 bounded heap 或 Space-Saving；CMS 只负责频次 oracle。width=6 的受控碰撞仍保持真实 Top-3。

In [5]:
cms_ranking = sorted(((term, cms.estimate(term)[0]) for term in observed_terms), key=lambda pair: (-pair[1], pair[0]))  # 用候选集合查询 sketch 并生成估计排名
exact_top3 = [term for term, count in exact_ranking[:3]]  # 取得精确基线的三个最热查询
cms_top3 = [term for term, count in cms_ranking[:3]]  # 取得 CMS 估计的三个最热查询
comparison = [{"搜索词": row["搜索词"], "真实频次": row["真实频次"], "CMS估计": row["CMS估计"], "误差": row["高估误差"], "真实排名": next(index + 1 for index, pair in enumerate(exact_ranking) if pair[0] == row["搜索词"]), "CMS排名": next(index + 1 for index, pair in enumerate(cms_ranking) if pair[0] == row["搜索词"])} for row in estimate_rows]  # 构造逐 key 频次和排名对照
print("Exact 与 CMS 排名对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示固定内存近似对每个搜索词的影响
print({"Exact Top3": exact_top3, "CMS Top3": cms_top3, "固定counter数": cms.width * cms.depth, "出现过的key数": len(observed_terms)})  # 汇总 Top-K 与内存规模

Exact 与 CMS 排名对照：
[{'搜索词': '退款', '真实频次': 20, 'CMS估计': 20, '误差': 0, '真实排名': 1, 'CMS排名': 1},
 {'搜索词': '订单', '真实频次': 15, 'CMS估计': 15, '误差': 0, '真实排名': 2, 'CMS排名': 2},
 {'搜索词': '发票', '真实频次': 10, 'CMS估计': 12, '误差': 2, '真实排名': 3, 'CMS排名': 3},
 {'搜索词': '密码', '真实频次': 7, 'CMS估计': 7, '误差': 0, '真实排名': 4, 'CMS排名': 4},
 {'搜索词': '物流', '真实频次': 5, 'CMS估计': 7, '误差': 2, '真实排名': 5, 'CMS排名': 5},
 {'搜索词': '天气', '真实频次': 4, 'CMS估计': 4, '误差': 0, '真实排名': 6, 'CMS排名': 8},
 {'搜索词': 'RAG', '真实频次': 3, 'CMS估计': 5, '误差': 2, '真实排名': 7, 'CMS排名': 6},
 {'搜索词': '向量', '真实频次': 2, 'CMS估计': 5, '误差': 3, '真实排名': 8, 'CMS排名': 7}]
{'Exact Top3': ['退款', '订单', '发票'], 'CMS Top3': ['退款', '订单', '发票'], '固定counter数': 18, '出现过的key数': 8}


## 6. 失败案例与修正：重复负更新破坏无关 key 的计数

标准 CMS 的只高估保证建立在非负 append-only 更新上。若“订单”删除事件重放两次，每次都把三个共享桶减 15，不仅订单变负，还会让发票、密码、天气等无关估计变负。修正是不要在标准 CMS 原地删除：按时间窗轮换，并从带 event_id 的 append 日志重建当前窗口。

In [6]:
broken_cms = CountMinSketch(width=6, depth=3)  # 创建用于复现负更新故障的独立 sketch
for term in stream:  # 先写入与主实验完全相同的搜索事件
    broken_cms.add(term)  # 建立初始正确的非负计数矩阵
broken_cms.add("订单", count=-frequencies["订单"])  # 第一次删除订单整个聚合计数
broken_cms.add("订单", count=-frequencies["订单"])  # 模拟非幂等 delete 消息重放再次扣减
broken_estimates = {term: broken_cms.estimate(term)[0] for term in frequencies}  # 查询重复负更新后全部 key 的受污染估计
remaining_stream = [term for term in stream if term != "订单"]  # 从 append 事件日志构造不含已删除 key 的新时间窗
rebuilt_cms = CountMinSketch(width=6, depth=3)  # 为新窗口创建空 sketch 避免原地负更新
for term in remaining_stream:  # 从受控事件日志重放仍然有效的正更新
    rebuilt_cms.add(term)  # 只使用 append 操作重建固定内存计数
rebuilt_estimates = {term: rebuilt_cms.estimate(term)[0] for term in frequencies if term != "订单"}  # 读取重建后其余 key 的非负估计
print({"失败_重复负更新估计": broken_estimates, "出现负数的key": [term for term, value in broken_estimates.items() if value < 0], "修正_窗口重建估计": rebuilt_estimates})  # 展示跨 key 污染与 append-only 重建修正

{'失败_重复负更新估计': {'退款': 20, '订单': -15, '发票': -5, '密码': -4, '物流': 7, '天气': -4, 'RAG': 5, '向量': 5}, '出现负数的key': ['订单', '发票', '密码', '天气'], '修正_窗口重建估计': {'退款': 20, '发票': 10, '密码': 7, '物流': 7, '天气': 4, 'RAG': 5, '向量': 5}}


## 7. 生产差距与最小回归检查

生产 CMS 要根据允许误差选择 width/depth，使用高质量独立哈希，并对 counter overflow、并发原子更新与跨分片逐桶相加做验证。Top-K 候选结构本身也有内存和误差；滑动窗口通常用多代 sketch 而非删除旧事件。下面的断言只验证本实验的固定矩阵、只高估、Top-3、碰撞和负更新失败。

In [7]:
assert len(frequencies) >= 6 and len(stream) == sum(frequencies.values())  # 确认真实搜索词和流事件规模满足教学要求
assert cms.width * cms.depth == 18  # 确认 sketch 内存固定为十八个 counters
assert all(row["CMS估计"] >= row["真实频次"] for row in estimate_rows)  # 确认非负流上的 Count-Min 估计不会低于真实频次
assert any(row["高估误差"] > 0 for row in estimate_rows)  # 确认受控宽度真实产生可观察哈希碰撞
assert cms_top3 == exact_top3  # 确认本例固定内存估计仍保留真实热门查询 Top-3
assert any(value < 0 for value in broken_estimates.values())  # 确认重复负更新真实破坏 append-only 计数保证
assert all(value >= 0 for value in rebuilt_estimates.values())  # 确认从正事件日志重建恢复合法非负估计
print("回归检查通过：CMS 哈希桶、碰撞高估、Top-K 与负更新边界均已验证。")  # 输出最终验收结论

回归检查通过：CMS 哈希桶、碰撞高估、Top-K 与负更新边界均已验证。
